
# <font color="green">Receiving many parameters</font>

## Problem

* Write a function `many_args` that takes **twenty** `long` parameters but uses only two of them: it returns `a08 * 1000 + a15`, __in assembly__.
* That is, translate the following C function into assembly:
```
long many_args(long a00, long a01, long a02, long a03, long a04,
                long a05, long a06, long a07, long a08, long a09,
                long a10, long a11, long a12, long a13, long a14,
                long a15, long a16, long a17, long a18, long a19) {
  return a08 * 1000 + a15;
}
```
* The first eight parameters (`a00`–`a07`) arrive in registers `x0`–`x7`. The ones you need here are **passed on the stack**, because they come after the first eight:
  * `a08` is the first stack argument, at `[sp]`;
  * `a15` is the eighth stack argument, at `[sp, 56]` (that is, `[sp, 8*(15-8)]`).
* So you only need to load those two with `ldr`, do one multiply and one add, and return. Be careful to use the **correct** offsets --- using the wrong argument will give the wrong answer.
* Fill in the skeleton `many_args.s` (after `// ------- write your answer here -------`).
* The checker `check_many_args.c` verifies your result. If you see `OK`s and no errors, you are done.

## Hints

* There are only so many registers, so you cannot pass an arbitrary number of parameters in registers.
* The ARM64 ABI passes the first eight integer arguments in `x0`–`x7`; the ninth and later arrive on the **stack**, at `[sp]`, `[sp, 8]`, `[sp, 16]`, ... as seen on entry to the function. So `a08` is at `[sp]`, `a09` at `[sp, 8]`, ..., `a15` at `[sp, 56]`.
* The *Observe* cells below contain a simple example, `sum_last2`, which returns the sum of its 9th and 10th parameters (`a8` at `[sp]`, `a9` at `[sp, 8]`). Compile it and see how stack-passed arguments are read with `ldr` --- then apply the same idea to the two arguments this problem needs.



# 1. AI Tutor
## 1-1. Prepare
* Your personal AI tutor is provided for questions and feedback.
* Execute the following cell before you use it.

In [ ]:
import heytutor

## 1-2. Examples
* A general question
```
%%hey
What does the `ldr` instruction do in ARM64?
```

* A hint on this specific problem
```
%%hey problem_file=many_args.md
Give me a hint on this problem.

{problem}
```

* Builtin variables usable in `%%hey` cells
  * `{file:FILENAME}` is the content of FILE
  * `{bash[-1]}` is the output of the last `%%bash_` cell, `{bash[-2]}` the second last, etc.
  * `{problem}` is the content of the file you specify by `%%hey problem_file=foo.md`
  * `{answer}` is the content of the file you specify by `%%hey answer_file=foo.s`


# 2. Observe: compile example functions
* Before writing your own assembly, it helps to see what the compiler generates for related example functions.
* Running the first cell below writes `explore.c` (some small example functions related to this problem).
* The second cell compiles it with `gcc -O3 -S` and prints the generated assembly.
* Feel free to edit `explore.c` (change the code, add functions, change constants) and re-run the two cells to see how the assembly changes.

In [ ]:
%%writefile_ explore.c
/* Receiving parameters that do not fit in registers. The ARM64 ABI passes the
   first eight integer arguments in x0..x7; the ninth and later arrive on the
   STACK, at [sp], [sp,#8], ... as seen on entry. This simple example reads the
   9th and 10th parameters (a8 at [sp], a9 at [sp,#8]) with ldr. */
long sum_last2(long a0, long a1, long a2, long a3, long a4,
               long a5, long a6, long a7, long a8, long a9) {
  return a8 + a9;
}

In [ ]:
%%bash_
gcc -O3 -S explore.c
cat explore.s


# 3. Your Answer (assembly)
* Running the cell below writes the skeleton assembly file `many_args.s`.
* Fill in your instructions after the line `// ------- write your answer here -------`, then run the cell again to save it.

In [ ]:
%%writefile_ many_args.s
	.arch armv8-a
	.file	"many_args.c"
	.text
	.align	2
	.p2align 4,,11
	.global	many_args
	.type	many_args, %function
many_args:
.LFB0:
	.cfi_startproc
	// ------- write your answer here -------
	.cfi_endproc
.LFE0:
	.size	many_args, .-many_args
	.ident	"GCC: (Ubuntu 13.3.0-6ubuntu2~24.04) 13.3.0"
	.section	.note.GNU-stack,"",@progbits


# 4. Checker
* The following C program calls your `many_args` function and checks the result against a reference computed in C.

In [ ]:
%%writefile_ check_many_args.c
#include <assert.h>
#include <stdio.h>
#include <stdlib.h>
long many_args(long a00, long a01, long a02, long a03, long a04, long a05, long a06, long a07, long a08, long a09, long a10, long a11, long a12, long a13, long a14, long a15, long a16, long a17, long a18, long a19);

int main(int argc, char ** argv) {
  assert(argc == 21);
  long a[20];
  for (int i = 0; i < 20; i++) a[i] = atol(argv[i + 1]);
  long r = many_args(a[0], a[1], a[2], a[3], a[4], a[5], a[6], a[7], a[8], a[9],
                      a[10], a[11], a[12], a[13], a[14], a[15], a[16], a[17], a[18], a[19]);
  long rc = a[8] * 1000 + a[15];   /* a08 * 1000 + a15 */
  if (r == rc) { printf("OK %ld %ld\n", r, rc); return 0; }
  else { printf("NG %ld %ld\n", r, rc); return 1; }
}


# 5. Compile
* Compile your assembly together with the checker.
* If you get an error, fix `many_args.s` above and recompile.

In [ ]:
%%bash_
gcc -o check_many_args -O3 check_many_args.c many_args.s -lm


# 6. Run
* If you see `OK`s and no errors, you are done.

In [ ]:
%%bash_
./check_many_args 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20
./check_many_args 0 0 0 0 0 0 0 0 7 0 0 0 0 0 0 -3 0 0 0 0


# 7. If things do not go well
* If your program compiles but does not produce the correct answer, run it within a debugger (gdb).
* Compile with `-O0 -g` first:
```
gcc -o check_many_args -O0 -g check_many_args.c many_args.s -lm
```
* Then, in a terminal (SSH or Jupyter terminal):
```
gdb check_many_args
(gdb) break many_args
(gdb) run ...        # give the same arguments as above
```
* Step through one instruction at a time with `step`, and inspect registers with `print $x0` or `info registers`.

# 8. Ask Questions or Get Feedback
* You are encouraged to ask for feedback once you think you are done, to know if there is a better answer.

In [ ]:
%%hey problem_file=many_args.md answer_file=many_args.s

Problem:
{problem}

My Answer:
{answer}

Give me a feedback to my answer.